# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 4 — CTR / Engagement Opportunity Scoring.**


Two audits, same four questions each time: **what is the main idea, where does the label come
from, what does the test design really check, and which claims does the evidence carry?**

| Card asks for | Lives in |
|---|---|
| Two paper findings + the methodology question for each | §1 |
| My model under an honest split, before/after | §2 |
| Leakage audit on the final feature set | §3 |
| My boldest claim, rewritten in safe language | §4 |

**Two rules I hold to.** *A check looks at the exact data that produced the claim*, so §2 reruns
Week 5 unchanged before it changes anything, and the reproduction is asserted against the committed
receipt. And *the paper is not being graded*, it was written for a broad audience, discloses its
own standards on page 2 and page 36, and my job is to practise the next level of rigour on my own
work.

## 0. Setup — three months, because this week adds a time axis

Week 5 used two months: March features, April outcome. This week adds **February**, so the model
can be trained at one decision moment and deployed unchanged at the next.

```text
  TRAIN FRAME                          DEPLOY FRAME
  Feb features -> Mar outcome          Mar features -> Apr outcome
  |-----------|  |-----------|         |-----------|  |-----------|
  02-01  02-28   03-01  03-31          03-01  03-31   04-01  04-30
              |                                    |
     decision moment 03-01                decision moment 04-01        [June SEALED]

In [1]:
%pip -q install --upgrade duckdb

import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
print("token loaded:", bool(HF_TOKEN))

token loaded: True


In [2]:
import duckdb, pandas as pd, numpy as np, sklearn
pd.set_option("display.width", 165); pd.set_option("display.max_columns", 60)
SEED = 42
print(f"duckdb {duckdb.__version__} | sklearn {sklearn.__version__} | pandas {pd.__version__}")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL  = "hf://datasets/FlyRank/internship-warehouse"
DIMC = f"read_parquet('{REL}/dim_content.parquet')"
def month_rel(m): return f"read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')"

# Contract constants -- unchanged from ML-04 through ML-08.
MIN_IMPRESSIONS, MIN_ACTIVE_DAYS, MIN_POSITION = 500, 5, 1.0
GAP_MIN, CLICKS_MIN, OUT_MIN_IMPRESSIONS       = 0.10, 10, 100

# Each month gets its own half-window dates, because February is short. Using -18/-04 for a
# 28-day month would compare 11 days against 14 and call it momentum.
HALVES = {"2026-02": ("2026-02-01", "2026-02-15"),
          "2026-03": ("2026-03-04", "2026-03-18"),
          "2026-04": ("2026-04-03", "2026-04-17")}

duckdb 1.5.5 | sklearn 1.6.1 | pandas 2.2.3


In [3]:
def page_month(month):
    prev_from, last_from = HALVES[month]
    df = con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                                AS impressions,
               SUM(gsc_clicks)                                                     AS clicks,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)  AS active_days,
               MAX(gsc_impressions)                                                AS top_day_impressions,
               SUM(gsc_sum_position) FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS pos_num,
               SUM(gsc_impressions)  FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS pos_den,
               STDDEV_SAMP(gsc_avg_position) FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS position_volatility,
               SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{last_from}') AS imp_last_half,
               SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{prev_from}'
                                              AND report_date <  DATE '{last_from}') AS imp_prev_half,
               MIN(report_date) AS first_date, MAX(report_date) AS last_date
        FROM {month_rel(month)}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        ORDER BY client_hash_id, content_hash_id
    """).df()
    df["avg_position"] = df.pos_num / df.pos_den.replace(0, np.nan) + 1   # zero-based fix, ML-04
    df["ctr_pp"]       = 100 * df.clicks / df.impressions.replace(0, np.nan)
    return df

def add_gap(df, tag):
    """Leave-one-out, volume-weighted peer baseline inside position tier. ML-04's definition."""
    df = df.copy()
    df["tier"] = df.avg_position.apply(
        lambda p: "top_3" if p <= 3 else "page_1" if p <= 10 else "striking" if p <= 20
        else "page_3_5" if p <= 50 else "deep")
    tc, ti = df.groupby("tier").clicks.transform("sum"), df.groupby("tier").impressions.transform("sum")
    df[f"{tag}_peer_pp"]       = (100 * (tc - df.clicks) /
                                  (ti - df.impressions)).replace([np.inf, -np.inf], np.nan)
    df[f"{tag}_gap_pp"]        = df[f"{tag}_peer_pp"] - df.ctr_pp
    df[f"{tag}_missed_clicks"] = df.impressions * df[f"{tag}_gap_pp"] / 100
    return df

dimc = con.sql(f"""SELECT content_hash_id, word_count, content_type, content_created_date
                   FROM {DIMC}""").df()
months = {m: page_month(m) for m in ["2026-02", "2026-03", "2026-04"]}
for m, d in months.items():
    print(f"{m}: {len(d):>7,} pages | dates {str(d.first_date.min())[:10]} .. {str(d.last_date.max())[:10]}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-02: 153,559 pages | dates 2026-02-01 .. 2026-02-28
2026-03: 176,738 pages | dates 2026-03-01 .. 2026-03-31
2026-04: 194,760 pages | dates 2026-04-01 .. 2026-04-30


In [4]:
SHAPE = ["log_impressions", "active_days", "position_volatility", "top_day_share",
         "momentum_log", "impressions_share_of_client", "log_word_count", "content_meta_missing"]
PRIOR = ["gap_pp", "ctr_pp_feat", "log_clicks"]

def build_frame(feature_month, outcome_month):
    """Features from one month, label from the NEXT month. Nothing crosses the line."""
    f = months[feature_month]
    f = f[(f.impressions >= MIN_IMPRESSIONS) & (f.active_days >= MIN_ACTIVE_DAYS) &
          (f.avg_position >= MIN_POSITION)].copy()
    n_eligible = len(f)
    f = add_gap(f, "f")

    f["log_impressions"] = np.log1p(f.impressions)
    f["top_day_share"]   = f.top_day_impressions / f.impressions
    f["momentum_log"]    = np.log((f.imp_last_half.fillna(0) + 1) / (f.imp_prev_half.fillna(0) + 1))
    f["impressions_share_of_client"] = (f.impressions /
                                        f.groupby("client_hash_id").impressions.transform("sum"))
    f = f.merge(dimc, on="content_hash_id", how="left", validate="many_to_one")
    wc = pd.to_numeric(f.word_count, errors="coerce").astype("float64")
    f["content_meta_missing"] = wc.isna().astype(int)
    f["log_word_count"] = np.log1p(wc.fillna(wc.groupby(f.client_hash_id).transform("median"))
                                     .fillna(wc.median()))
    f["gap_pp"], f["ctr_pp_feat"], f["log_clicks"] = f.f_gap_pp, f.ctr_pp, np.log1p(f.clicks)

    o = months[outcome_month]
    o = o[(o.impressions >= OUT_MIN_IMPRESSIONS) & (o.avg_position >= MIN_POSITION)].copy()
    o = add_gap(o, "o")
    o["label"] = ((o.o_gap_pp >= GAP_MIN) & (o.o_missed_clicks >= CLICKS_MIN)).astype(int)

    out = f.merge(o[["content_hash_id", "label", "o_gap_pp", "impressions", "avg_position"]]
                    .rename(columns={"impressions": "out_impressions",
                                     "avg_position": "out_position"}),
                  on="content_hash_id", how="inner", validate="one_to_one")
    out = out.dropna(subset=SHAPE + PRIOR + ["label"]).copy()
    out.attrs["n_eligible"] = n_eligible
    return out

train_df  = build_frame("2026-02", "2026-03")    # decision moment 2026-03-01
deploy_df = build_frame("2026-03", "2026-04")    # decision moment 2026-04-01 (the ML-08 frame)

for name, d in [("TRAIN  (Feb->Mar)", train_df), ("DEPLOY (Mar->Apr)", deploy_df)]:
    print(f"{name}: {len(d):>7,} rows | {d.client_hash_id.nunique():>2} clients | "
          f"base rate {d.label.mean():.3f} | eligible before outcome join {d.attrs['n_eligible']:,}")

TRAIN  (Feb->Mar):  45,931 rows | 31 clients | base rate 0.119 | eligible before outcome join 47,058
DEPLOY (Mar->Apr):  60,942 rows | 34 clients | base rate 0.095 | eligible before outcome join 61,881


In [5]:
# --- Reproduce Week 5 BEFORE changing anything -------------------------------
W5 = {"march_eligible": 61881, "rows": 60942, "clients": 34, "base_rate": 0.095}
got = {"march_eligible": deploy_df.attrs["n_eligible"], "rows": len(deploy_df),
       "clients": deploy_df.client_hash_id.nunique(), "base_rate": round(deploy_df.label.mean(), 3)}
print("reproduction check against the committed ML-08 receipt")
for k, v in W5.items():
    print(f"  {k:16} receipt {str(v):>7}   this run {str(got[k]):>7}   "
          f"{'MATCH' if v == got[k] else 'DIFFERS'}")
assert got["rows"] == W5["rows"], "the Week-5 frame did not reproduce -- stop and investigate"
print("\nPASS: the deploy frame is the Week-5 frame, rebuilt from source. Everything below is a "
      "change to the TEST, not to the data.")

reproduction check against the committed ML-08 receipt
  march_eligible   receipt   61881   this run   61881   MATCH
  rows             receipt   60942   this run   60942   MATCH
  clients          receipt      34   this run      34   MATCH
  base_rate        receipt   0.095   this run   0.095   MATCH

PASS: the deploy frame is the Week-5 frame, rebuilt from source. Everything below is a change to the TEST, not to the data.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Both questions asked of both findings: **where does the label come from**, and **does the
validation design carry the claim?** Framed the way I would want my own work reviewed — the paper
states its standards on page 2 (clarity over the biggest claim) and page 36 (no p-values or
confidence intervals), so nothing below is a gotcha.

I deliberately picked two findings the live session did **not** walk through, so this is my own
reading rather than a replay.

---

### Finding A — "Anatomy of growing content" (Finding #1)

**What it claims.** Growing pages are 37.6% longer (3,180 vs 2,311 words) and about 20% younger
(184 vs 230 days) than declining pages. Samples are large: 74.8K rising, 45.6K declining.

**Where does the label come from?** "Rising" and "declining" are a *stored trend direction*, not
something the paper derives in front of the reader. Three things I would want stated:

1. **Over what window** is the trend measured — 30 days, 90 days, the full panel?
2. **What is the threshold** for rising versus declining, and what happens to pages that are flat?
   If flat pages are dropped, the two groups are the extremes of a distribution rather than its two
   halves, and every gap between them is amplified by construction.
3. **Is it the same field my own release calls `trend_direction`?** If so, my Week-1 notes already
   flag it as label-derived — fine as an outcome, but its definition then drives the whole finding.

**Does the validation design carry the claim?** The comparison is a **two-group average with no
control**. The paper is careful to call it observational, which is right. My question is about
**confounding**: growing pages are *also* younger, and Finding #2 shows health peaks at 61–90 days.
So "growth", "youth" and "lifecycle stage" all move together, and a length gap between the groups
could be a length effect, an age effect, or a lifecycle effect wearing a length costume.

**What would make it stronger** (and this is cheap): report the word-count gap **within age bands**.
If longer pages still grow more among 61–90-day pages *and* among 271–365-day pages, length is
carrying its own weight. If the gap disappears inside bands, age was doing the work. Also report the
**median and the distribution**, not only the mean — one bucket of 8,000-word pillar pages moves a
mean a long way.

I run exactly that check on my own data below, because the question is only fair if I am willing to
answer it about myself.

---

### Finding B — "Old refreshed content ≈ new content" (Finding #8)

**What it claims.** Old content that gets refreshed scores 44.62 health, against 44.12 for new
content — practically identical. It is the paper's most actionable finding: refreshing works.

**Where does the label come from?** The outcome here is **Health Score**, which page 5 defines as a
hand-built recipe: impressions 30 points, position 30, CTR 20, scroll depth 20. That is not a
neutral measure of quality — it is FlyRank's own summary, and the paper says so. The consequence:
"refreshed old pages perform as well as new pages" means *"they score as well on a formula that is
60% impressions and position."* A refreshed page that regains impressions scores well almost by
definition.

**Does the validation design carry the claim?** This is the one I would raise first, and the paper
half-raises it itself — it warns that one cell of the matrix carries "strong survivor bias" from the
active-content subset. My question is whether that warning is **scoped too narrowly**, because the
same filter appears to run under the whole section:

> the "active content" sample keeps pages with impressions **and** sessions in the window.

Old pages that were *never* refreshed and lost all their traffic have no impressions — so they are
**not in the comparison at all**. The surviving stale pages are the healthiest stale pages. That
makes 44.62 vs 44.12 a comparison between refreshed old pages and *the stale old pages that
survived*, which is a different and much weaker statement than "refreshing works."

**What would make it stronger:** state the population filter next to the finding rather than in a
methods note, report how many old pages were excluded by it, and — best of all — track a fixed
cohort forward (pages alive in month 1, refreshed or not, followed to month 6) so the ones that die
stay countable as zeros instead of vanishing.

**And the honest mirror:** my own Week-5 population has the same shape of problem, disclosed in §3
below. I drop pages with no readable outcome data. That is a choice, not a crime — but it is the
same choice, and I should not ask of a paper what I do not do myself.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Finding A's question, asked of MY OWN data ------------------------------
# Claim shape: "persistent pages are longer than recovered pages". Then the control: does the
# gap survive INSIDE an age band, or was age doing the work?
d = deploy_df.copy()
d["created"] = pd.to_datetime(d.content_created_date, errors="coerce")
d["age_days"] = (pd.Timestamp("2026-03-31") - d["created"]).dt.days
d["words"] = pd.to_numeric(d.word_count, errors="coerce")
have = d.dropna(subset=["age_days", "words"])
print(f"pages with both word count and creation date: {len(have):,} of {len(d):,} "
      f"({len(have)/len(d):.1%})\n")

print("UNCONTROLLED -- the two-group comparison, the shape the paper reports")
u = have.groupby("label").agg(n=("words", "size"), mean_words=("words", "mean"),
                              median_words=("words", "median"), median_age=("age_days", "median"))
display(u.round(1))
gap_all = u.loc[1, "median_words"] - u.loc[0, "median_words"]
print(f"median word gap, persistent minus recovered: {gap_all:+.0f} words")

print("\nCONTROLLED -- the same gap INSIDE age bands (>=50 rows per cell, ML-06's floor)")
have = have.copy()
have["age_band"] = pd.cut(have.age_days, [-1, 90, 180, 365, 10**6],
                          labels=["<=90d", "91-180d", "181-365d", "365d+"])
rows = []
for band, sub in have.groupby("age_band", observed=True):
    a, b = sub[sub.label == 1], sub[sub.label == 0]
    if len(a) >= 50 and len(b) >= 50:
        rows.append({"age_band": band, "n_persistent": len(a), "n_recovered": len(b),
                     "median_words_persistent": a.words.median(),
                     "median_words_recovered":  b.words.median(),
                     "gap": a.words.median() - b.words.median()})
ctrl = pd.DataFrame(rows)
display(ctrl.round(1))
if len(ctrl):
    same_sign = (np.sign(ctrl.gap) == np.sign(gap_all)).sum()
    print(f"\nOBSERVED: the uncontrolled gap is {gap_all:+.0f} words. Inside age bands it keeps the "
          f"same direction in {same_sign} of {len(ctrl)} bands "
          f"(band gaps: {', '.join(f'{g:+.0f}' for g in ctrl.gap)}).")
    print("This is the check I would ask the paper for, run on my own numbers first. A gap that "
          "survives inside bands is a length effect; one that vanishes was an age effect.")

pages with both word count and creation date: 48,794 of 60,942 (80.1%)

UNCONTROLLED -- the two-group comparison, the shape the paper reports


,n,mean_words,median_words,median_age
label,,,,
0,43963,3046.1,2794.0,158.0
1,4831,2838.6,2712.0,166.0


median word gap, persistent minus recovered: -82 words

CONTROLLED -- the same gap INSIDE age bands (>=50 rows per cell, ML-06's floor)


,age_band,n_persistent,n_recovered,median_words_persistent,median_words_recovered,gap
0,<=90d,1835,16754,2747.0,2854.0,-107.0
1,91-180d,675,7818,2702.0,3070.5,-368.5
2,181-365d,1686,15704,2694.5,2694.0,0.5
3,365d+,635,3687,2662.0,2734.0,-72.0



OBSERVED: the uncontrolled gap is -82 words. Inside age bands it keeps the same direction in 3 of 4 bands (band gaps: -107, -368, +0, -72).
This is the check I would ask the paper for, run on my own numbers first. A gap that survives inside bands is a length effect; one that vanishes was an age effect.


**What that check actually returned — and it is not what I framed it as.**

The uncontrolled gap is **−82 words**: pages whose CTR gap persisted into April are *shorter* than
pages that recovered, not longer. Inside age bands the direction holds in 3 of 4 (−107, −368, ~0,
−72), so it is not an age effect wearing a length costume — which is exactly the control I asked
the paper to run.

**Two things this does not mean.** First, it does not contradict the paper's Finding #1. Their
comparison is *growing vs declining traffic*; mine is *persistent underperformance vs recovery*.
Different labels, different questions — and noticing that they are different is half the point of
this section. Second, −82 words on a median near 2,750 is a ~3% difference, small enough that I
would not lead with it in either direction.

**What it does show** is that the control changes what you would say. The uncontrolled number and
the banded numbers agree here, so the finding is stable — but I only know that because I ran it,
and the paper's Finding #1 reports the uncontrolled version only.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Week 5 already used a grouped split** — GroupKFold on client, every metric out-of-fold, with an
assertion that no client appears on both sides. So "grouped" is the *before*, not the improvement.

**The improvement this week is time-aware**, and it is the harder test:

| | question it answers | Week 5 | this week |
|---|---|---|---|
| grouped by client | does it work on a client we have never seen? | ✓ | ✓ (re-run) |
| **time-forward** | does a model trained *last* month still work *this* month, deployed unchanged? | ✗ | **✓ new** |

**The design.** Train once on the February→March frame. Freeze it. Apply it to the March→April
frame with no refitting, no threshold tuning, nothing. That is what deployment actually looks like:
you fit on what you had, and the next month arrives whether you are ready or not.

**Everything is scored on the same deploy rows, with the same metric and the same K**, so the two
numbers are comparable — the session's comparison contract.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

FEATS = SHAPE + PRIOR
def rf(): return RandomForestClassifier(n_estimators=60, max_depth=10, min_samples_leaf=40,
                                        random_state=SEED, n_jobs=1)

def precision_at_k(scores, y_true, k, tie_seed=SEED):
    """Ties broken by seeded jitter, so file order never decides the queue."""
    s = np.asarray(scores, dtype=float)
    jitter = np.random.default_rng(tie_seed).random(len(s)) * 1e-12
    order = np.lexsort((jitter, -s))[:k]
    return float(np.mean(np.asarray(y_true)[order]))

y_dep  = deploy_df.label.values
grp    = deploy_df.client_hash_id.values
base   = y_dep.mean()

# --- BEFORE: grouped cross-validation inside the deploy month (the Week-5 design) ---
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(deploy_df))
X_dep = deploy_df[FEATS].values
for tr, te in gkf.split(X_dep, y_dep, grp):
    assert set(grp[tr]).isdisjoint(set(grp[te]))
    oof[te] = rf().fit(X_dep[tr], y_dep[tr]).predict_proba(X_dep[te])[:, 1]

# --- AFTER: trained one month earlier, deployed unchanged ---------------------
model_tf = rf().fit(train_df[FEATS].values, train_df.label.values)
p_tf     = model_tf.predict_proba(X_dep)[:, 1]

rows = [
    {"design": "0. random ranking (floor)",              "scores": np.random.default_rng(SEED).random(len(deploy_df))},
    {"design": "1. ML-07 rule, no learning",             "scores": deploy_df.f_missed_clicks.values},
    {"design": "2. BEFORE - grouped CV, same month",     "scores": oof},
    {"design": "3. AFTER  - trained Feb->Mar, deployed unchanged", "scores": p_tf},
]
tbl = pd.DataFrame([{"design": r["design"],
                     "precision@50":  round(precision_at_k(r["scores"], y_dep, 50), 3),
                     "precision@200": round(precision_at_k(r["scores"], y_dep, 200), 3),
                     "ROC AUC":       round(roc_auc_score(y_dep, r["scores"]), 3)} for r in rows])
tbl["lift@50"] = (tbl["precision@50"] / base).round(2)
print(f"All four scored on the SAME {len(deploy_df):,} deploy rows. BASE RATE = {base:.3f}\n")
display(tbl)

All four scored on the SAME 60,942 deploy rows. BASE RATE = 0.095



,design,precision@50,precision@200,ROC AUC,lift@50
0,0. random ranking (floor),0.04,0.060,0.500,0.42
1,"1. ML-07 rule, no learning",0.90,0.875,0.864,9.48
2,"2. BEFORE - grouped CV, same month",0.96,0.975,0.921,10.11
3,"3. AFTER - trained Feb->Mar, deployed unchanged",0.96,0.980,0.925,10.11


In [8]:
# --- What changed between training the model and using it? -------------------
tr_clients, dep_clients = set(train_df.client_hash_id), set(deploy_df.client_hash_id)
shared, new = len(tr_clients & dep_clients), len(dep_clients - tr_clients)
print("Drift between the training moment and the deployment moment")
print(f"  base rate when trained (Feb->Mar) : {train_df.label.mean():.3f}")
print(f"  base rate when used   (Mar->Apr)  : {base:.3f}   "
      f"({base - train_df.label.mean():+.3f})")
print(f"  training rows {len(train_df):,} / deploy rows {len(deploy_df):,}")
print(f"  deploy clients also seen in training: {shared} | never seen before: {new}")
print()
before_50, after_50 = tbl.loc[2, "precision@50"], tbl.loc[3, "precision@50"]
print(f"BEFORE (grouped CV, same month)      precision@50 = {before_50:.3f}")
print(f"AFTER  (trained a month earlier)     precision@50 = {after_50:.3f}   "
      f"({after_50 - before_50:+.3f})")
print(f"rule {tbl.loc[1, 'precision@50']:.3f} | random {tbl.loc[0, 'precision@50']:.3f} | "
      f"base rate {base:.3f}")
print()
print("OBSERVED: the time-forward number is the one that answers 'would this have worked if I had "
      "actually shipped it a month ago'. The grouped number answers a different and easier "
      "question -- a model that saw April-month patterns during cross-validation is not the same "
      "as a model fitted before April existed. Whichever way the gap falls, the honest headline "
      "is the time-forward number, and the base rate moved between the two moments, which is drift "
      "and not a bug.")

Drift between the training moment and the deployment moment
  base rate when trained (Feb->Mar) : 0.119
  base rate when used   (Mar->Apr)  : 0.095   (-0.024)
  training rows 45,931 / deploy rows 60,942
  deploy clients also seen in training: 30 | never seen before: 4

BEFORE (grouped CV, same month)      precision@50 = 0.960
AFTER  (trained a month earlier)     precision@50 = 0.960   (+0.000)
rule 0.900 | random 0.040 | base rate 0.095

OBSERVED: the time-forward number is the one that answers 'would this have worked if I had actually shipped it a month ago'. The grouped number answers a different and easier question -- a model that saw April-month patterns during cross-validation is not the same as a model fitted before April existed. Whichever way the gap falls, the honest headline is the time-forward number, and the base rate moved between the two moments, which is drift and not a bug.


In [9]:
# --- Is the time-forward margin over the rule real, or resampling noise? -----
rng = np.random.default_rng(SEED)
rule_scores, idx, boot = deploy_df.f_missed_clicks.values, np.arange(len(deploy_df)), []
for _ in range(300):
    s = rng.choice(idx, size=len(idx), replace=True)
    boot.append(precision_at_k(p_tf[s], y_dep[s], 200) - precision_at_k(rule_scores[s], y_dep[s], 200))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"time-forward model minus rule, precision@200: {np.mean(boot):+.3f} "
      f"(95% bootstrap interval {lo:+.3f} to {hi:+.3f}, 300 resamples)")
print("-> the interval EXCLUDES zero, so on this evidence the difference is unlikely to be "
      "resampling noise." if lo > 0 or hi < 0 else
      "-> the interval CONTAINS zero. I cannot claim a real difference at K=200 from one deployment "
      "window; reporting it as a tie is the honest call.")
print("\nLimit of this test, stated: it is ONE deployment window. A full backtest repeats it across "
      "many decision moments. This is evidence, not proof, and the five bootstrap-resampled folds "
      "are drawn from the same month, so they are related rather than independent.")

time-forward model minus rule, precision@200: +0.103 (95% bootstrap interval +0.055 to +0.155, 300 resamples)
-> the interval EXCLUDES zero, so on this evidence the difference is unlikely to be resampling noise.

Limit of this test, stated: it is ONE deployment window. A full backtest repeats it across many decision moments. This is evidence, not proof, and the five bootstrap-resampled folds are drawn from the same month, so they are related rather than independent.


**A reproducibility finding, discovered by re-running.** With seeds fixed and `n_jobs=1`, repeated
runs still produced different numbers — time-forward precision@50 came out 0.98, then 0.96. The
cause was row order: the source query had no `ORDER BY`, and DuckDB does not guarantee ordering
across parallel parquet reads. Random Forest bootstraps rows *by index*, so a different sequence
builds slightly different trees, and seeded randomness lands on different rows.

**Adding `ORDER BY` fixed the instability — and moved the Week-5 headline.** With ordering pinned,
grouped cross-validation gives precision@50 of **0.96**, not the **1.00** recorded in the ML-08
receipt. The difference is two pages in fifty, and it does not change any conclusion: the model
still beats the rule at 0.90, and the bootstrap interval on the margin still excludes zero.

But it does change what I am entitled to say. "Perfect at the top fifty" was partly an artifact of
row ordering, and I only know that because this audit re-ran the thing rather than quoting it. The
receipt itself was honest — it recorded what that run produced. It was the *stability* of that
number I had not tested. **Observed, measured, and now reproducible.**

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The skill's attack checklist, run item by item on the final feature set — plus the test the session
added this week: **deliberately inject a leak and confirm the detector fires.** A clean result from
a broken detector is worth nothing.

### The timeline

```text
features may use ────────────► 2026-03-31
                                    │  decision moment 2026-04-01
outcome window ─────────────────────► 2026-04-01 .. 2026-04-30
```

Every one of the 11 features is aggregated from `report_date <= 2026-03-31`. April clicks build the
label. **April impressions do not appear in the label formula — and they are still excluded**,
because they come from the outcome month, and being innocent is not the same as being on the right
side of the line.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("ATTACK CHECKLIST\n" + "=" * 78)

# 1. Timeline, verified in code rather than from memory
feat_last  = str(months["2026-03"].last_date.max())[:10]
label_first = str(months["2026-04"].first_date.min())[:10]
c1 = feat_last < label_first
print(f"[{'PASS' if c1 else 'FAIL'}] timeline: last feature date {feat_last} is before first "
      f"outcome date {label_first}")

# 2. No label-derived or sibling columns among the features
LABEL_SIDE = {"label", "o_gap_pp", "o_peer_pp", "o_missed_clicks", "out_impressions", "out_position"}
c2 = set(FEATS).isdisjoint(LABEL_SIDE)
print(f"[{'PASS' if c2 else 'FAIL'}] no outcome-month column is a feature "
      f"{set(FEATS) & LABEL_SIDE if not c2 else ''}")

# 3. No product flags / existing-system scores
PRODUCT = {"health_score", "needs_ctr_fix", "is_quick_win", "trend_direction", "trend_pct"}
c3 = PRODUCT.isdisjoint(set(deploy_df.columns))
print(f"[{'PASS' if c3 else 'FAIL'}] no FlyRank product flag anywhere in the frame "
      f"(the ML-07 rule is a BASELINE to beat, never an input)")

# 4. Population selection checked for outcome-window information -- DISCLOSED, not clean
dropped = deploy_df.attrs["n_eligible"] - len(deploy_df)
print(f"[DISCLOSED] population: {dropped:,} of {deploy_df.attrs['n_eligible']:,} eligible pages "
      f"({dropped/deploy_df.attrs['n_eligible']:.1%}) have no readable April data and were dropped.")
print(f"            That uses outcome-window information to define who is studied. It is a choice, "
      f"not a crime -- but it is the SAME choice I questioned in the paper's Finding #8, so it is "
      f"stated here and in section 4 rather than buried.")

# 5. Split grouped by the repeating entity
print(f"[PASS] split: GroupKFold on client_hash_id, disjointness asserted in section 2")

# 6. Base rate printed next to every metric
print(f"[PASS] base rate {base:.3f} printed beside every number in the section-2 table")

# 7 + 8 handled in the two cells below
print(f"[see below] top-feature sanity check, and the sabotage test")

ATTACK CHECKLIST
[PASS] timeline: last feature date 2026-03-31 is before first outcome date 2026-04-01
[PASS] no outcome-month column is a feature 
[PASS] no FlyRank product flag anywhere in the frame (the ML-07 rule is a BASELINE to beat, never an input)
[DISCLOSED] population: 939 of 61,881 eligible pages (1.5%) have no readable April data and were dropped.
            That uses outcome-window information to define who is studied. It is a choice, not a crime -- but it is the SAME choice I questioned in the paper's Finding #8, so it is stated here and in section 4 rather than buried.
[PASS] split: GroupKFold on client_hash_id, disjointness asserted in section 2
[PASS] base rate 0.095 printed beside every number in the section-2 table
[see below] top-feature sanity check, and the sabotage test


In [11]:
# --- The sabotage test: does my detector actually detect anything? -----------
# Inject a column the model must NOT be allowed: April impressions, from the outcome month.
X_clean  = deploy_df[FEATS].values
X_leaky  = np.column_stack([X_clean, np.log1p(deploy_df.out_impressions.values)])

def grouped_oof(X):
    o = np.zeros(len(deploy_df))
    for tr, te in gkf.split(X, y_dep, grp):
        o[te] = rf().fit(X[tr], y_dep[tr]).predict_proba(X[te])[:, 1]
    return o

p_clean, p_leaky = grouped_oof(X_clean), grouped_oof(X_leaky)
pc, pl = precision_at_k(p_clean, y_dep, 50), precision_at_k(p_leaky, y_dep, 50)
ac, al = roc_auc_score(y_dep, p_clean), roc_auc_score(y_dep, p_leaky)
print(f"clean features            : precision@50 {pc:.3f} | AUC {ac:.3f}")
print(f"+ April impressions (LEAK): precision@50 {pl:.3f} | AUC {al:.3f}")
print(f"the injected leak buys {pl - pc:+.3f} precision@50 and {al - ac:+.3f} AUC\n")
print("OBSERVED: the detector fires -- a planted outcome-month column visibly moves the score, so "
      "an all-clear from this setup means something. If the injected leak had changed nothing, the "
      "correct conclusion would be that my leak test is broken, not that my features are clean."
      if (al - ac) > 0.01 else
      "WARNING: the planted leak barely moved the score. That does not prove my features are "
      "clean -- it suggests this test is not sensitive enough to detect leakage at all, and the "
      "PASS marks above should not be trusted until I understand why.")
del X_leaky, p_leaky
print(f"\nNote on the two numbers: precision@50 could not move -- it was already at {pc:.2f}, a "
      f"ceiling. AUC had room and responded (+{al - ac:.3f}), which is why the verdict rests on "
      f"AUC. A saturated metric cannot detect anything, and that is worth knowing before trusting "
      f"any test built on it.")

clean features            : precision@50 0.960 | AUC 0.921
+ April impressions (LEAK): precision@50 1.000 | AUC 0.975
the injected leak buys +0.040 precision@50 and +0.054 AUC

OBSERVED: the detector fires -- a planted outcome-month column visibly moves the score, so an all-clear from this setup means something. If the injected leak had changed nothing, the correct conclusion would be that my leak test is broken, not that my features are clean.

Note on the two numbers: precision@50 could not move -- it was already at 0.96, a ceiling. AUC had room and responded (+0.054), which is why the verdict rests on AUC. A saturated metric cannot detect anything, and that is worth knowing before trusting any test built on it.


In [12]:
# --- Top-feature sanity: "too good" investigated, not celebrated -------------
from sklearn.inspection import permutation_importance
gkf_split = list(gkf.split(X_dep, y_dep, grp))[0]
tr, te = gkf_split
m = rf().fit(X_dep[tr], y_dep[tr])
pi = permutation_importance(m, X_dep[te], y_dep[te], n_repeats=5, random_state=SEED,
                            n_jobs=1, scoring="roc_auc")
imp = (pd.DataFrame({"feature": FEATS, "drop_in_auc": pi.importances_mean.round(4)})
       .sort_values("drop_in_auc", ascending=False).reset_index(drop=True))
display(imp)

ratio = imp.iloc[0].drop_in_auc / max(imp.iloc[1].drop_in_auc, 1e-9)
print(f"top feature {imp.iloc[0].feature} costs {imp.iloc[0].drop_in_auc:.4f} AUC, "
      f"{ratio:.1f}x the next")
if imp.iloc[0].feature == "log_impressions" and ratio > 2:
    print("Investigated in ML-08 and unchanged here: this is the LABEL's construction, not "
          "temporal leakage. The label needs April missed clicks >= 10, and missed clicks = "
          "impressions x gap, so page size is close to a prerequisite for a positive label. March "
          "impressions are knowable on 2026-04-01, so the feature is legal -- but the label is "
          "easier than it looks, and that belongs in the claim, not in a footnote.")
elif ratio > 2:
    print("One feature dominates and it is NOT the one ML-08 found. That is a change worth "
          "explaining before trusting anything above -- check whether this feature could carry "
          "outcome-window information, using the train-with / train-without test.")
else:
    print("No single feature dominates: several contribute at similar magnitude, which is the "
          "shape of weak signal used honestly rather than a model that found the answer. Note this "
          "differs from ML-08, where log_impressions towered over the rest -- worth one sentence "
          "on why the fold or the feature set changed the picture.")

,feature,drop_in_auc
0,log_impressions,0.1127
1,gap_pp,0.0483
2,ctr_pp_feat,0.0191
3,log_clicks,0.0190
4,position_volatility,0.0039
5,momentum_log,0.0038
6,impressions_share_of_client,0.0037
7,top_day_share,0.0008
8,active_days,0.0007
9,log_word_count,-0.0001


top feature log_impressions costs 0.1127 AUC, 2.3x the next
Investigated in ML-08 and unchanged here: this is the LABEL's construction, not temporal leakage. The label needs April missed clicks >= 10, and missed clicks = impressions x gap, so page size is close to a prerequisite for a positive label. March impressions are knowable on 2026-04-01, so the feature is legal -- but the label is easier than it looks, and that belongs in the claim, not in a footnote.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### My boldest sentence, as I could have written it after Week 5

> *"My model is 100% precise at the top fifty and beats the Week-4 rule, so it will find the pages
> worth reviewing for any client next month, and fixing them will stop their CTR gaps."*

Every clause in that sentence fails a check.

| the phrase | what is wrong with it |
|---|---|
| "100% precise" | true only for one K, on one month, under one split — and stated without the 9.5% base rate that makes it readable |
| "beats the Week-4 rule" | true, but the size of the win belongs in the sentence, not the fact of it |
| "for any client" | per-client AUC ran from **0.600 to 0.990**. It does not work equally for anyone |
| "next month" | tested on exactly **one** deployment window, and the base rate moved between training and use |
| "fixing them will stop their CTR gaps" | **a causal claim.** No page was edited, no control group existed, no experiment was run |

### The rewrite

> **Observed** on 60,942 pages across 34 clients, a Random Forest using March traffic shape and the
> March CTR gap **measured** precision@50 of **0.96 under grouped cross-validation and 0.96 under
> the stricter time-forward test** — trained on February→March and deployed unchanged — against a
> frozen Week-4 rule at 0.90 and a base rate of 0.095, with the margin at K=200 holding a bootstrap
> interval of +0.055 to +0.155 that excludes zero. That the two designs agree is worth stating: a
> model fitted before April existed did as well as one cross-validated inside April, which is the
> stronger of the two results. The improvement is **directional** rather than uniform: per-client
> AUC ranged from 0.600 to 0.990, the base rate moved from 0.119 to 0.095 between training and
> deployment, and one deployment window is not a backtest. The queue is **decision-support** for
> which pages an editor opens first — it does not identify what is wrong with a page, and nothing
> here shows that editing one changes its outcome.

### The effect size, which is the part people skip

Precision numbers do not tell an editor anything they can act on. **Pages** do. The cell below
converts the margin into the only unit that matters: how many more genuinely persistent pages an
editor opens per fifty reviewed.

### Three limits that travel with the claim

1. **The label is size-influenced by construction.** The ≥10-missed-clicks gate takes the base rate
   from 0.4% among small pages to 47.1% among large ones. Ranking large pages well is part of what
   is being measured.
2. **The label cannot separate "fixed" from "died."** Two of ML-08's three most confident errors
   were pages whose April traffic fell by more than half — the gap stopped clearing the threshold
   because demand left, not because the page improved.
3. **Survivor selection, same as the one I asked the paper about.** Pages with no readable April
   data are excluded. Disclosed in §3, and it may be removing exactly the pages that failed hardest.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- The claim, in units an editor can use -----------------------------------
for K in (50, 200):
    p_rule = precision_at_k(deploy_df.f_missed_clicks.values, y_dep, K)
    p_oof  = precision_at_k(oof,  y_dep, K)
    p_time = precision_at_k(p_tf, y_dep, K)
    print(f"top {K:>3} reviewed -> rule finds {p_rule*K:>5.0f} persistent pages | "
          f"grouped-CV model {p_oof*K:>5.0f} | time-forward model {p_time*K:>5.0f} | "
          f"random {base*K:>5.1f}")
print(f"\nEFFECT SIZE, stated plainly: against the frozen rule, the time-forward model changes an "
      f"editor's top-50 week by about "
      f"{(precision_at_k(p_tf, y_dep, 50) - precision_at_k(deploy_df.f_missed_clicks.values, y_dep, 50))*50:+.0f} "
      f"genuinely persistent pages out of fifty opened. That number, not a precision figure, is "
      f"what a reviewer should be told.")

top  50 reviewed -> rule finds    45 persistent pages | grouped-CV model    48 | time-forward model    48 | random   4.7
top 200 reviewed -> rule finds   175 persistent pages | grouped-CV model   195 | time-forward model   196 | random  19.0

EFFECT SIZE, stated plainly: against the frozen rule, the time-forward model changes an editor's top-50 week by about +3 genuinely persistent pages out of fifty opened. That number, not a precision figure, is what a reviewer should be told.


In [14]:
# --- Receipts ---------------------------------------------------------------
import json, os
os.makedirs("work/outputs", exist_ok=True)
receipts = {
    "assignment": "ML-09 - Validation and Research Claim Audit",
    "reproduction": {"week5_receipt": W5, "this_run": got, "matched": got["rows"] == W5["rows"]},
    "paper_findings_audited": [
        {"finding": "#1 anatomy of growing content",
         "label_question": "trend direction is a stored field: window, threshold and treatment of flat pages unstated",
         "design_question": "two-group averages with no control; growth, youth and lifecycle stage move together",
         "suggested_strengthening": "report the word gap within age bands, and medians alongside means"},
        {"finding": "#8 old refreshed content vs new content",
         "label_question": "outcome is Health Score, a hand-built recipe that is 60% impressions and position",
         "design_question": "the active-content filter removes old pages that lost all traffic, so the comparison is against surviving stale pages",
         "suggested_strengthening": "state the filter beside the finding, report how many pages it removes, follow a fixed cohort forward"},
    ],
    "honest_split": {"before": "GroupKFold(5) on client, same month",
                     "after": "trained Feb->Mar, deployed unchanged on Mar->Apr",
                     "table": tbl.to_dict(orient="records"),
                     "base_rate_train": round(float(train_df.label.mean()), 4),
                     "base_rate_deploy": round(float(base), 4),
                     "deploy_clients_seen_in_training": int(shared),
                     "deploy_clients_new": int(new)},
    "bootstrap_time_forward_vs_rule_at_200": {"mean": round(float(np.mean(boot)), 4),
                                              "ci95": [round(float(lo), 4), round(float(hi), 4)],
                                              "resamples": 300},
    "leakage_audit": {"timeline_ok": bool(c1), "no_outcome_columns": bool(c2),
                      "no_product_flags": bool(c3),
                      "population_dropped_no_outcome": int(dropped),
                      "sabotage_test_auc_gain": round(float(al - ac), 4),
                      "detector_fires": bool((al - ac) > 0.01)},
    "claim_language": ["observed", "measured", "directional", "decision-support"],
    "open_checks": ["only one deployment window - a full backtest needs many decision moments",
                    "no calibration check", "no causal evidence: nothing was intervened on",
                    "June 2026 remains sealed"],
}
with open("work/outputs/ml09_validation_audit_receipts.json", "w") as f:
    json.dump(receipts, f, indent=2, default=float)
print(tbl.to_string(index=False))
print(f"\nbase rate {base:.3f} | deploy rows {len(deploy_df):,} | clients {deploy_df.client_hash_id.nunique()}")
print("saved -> work/outputs/ml09_validation_audit_receipts.json")

                                          design  precision@50  precision@200  ROC AUC  lift@50
                       0. random ranking (floor)          0.04          0.060    0.500     0.42
                      1. ML-07 rule, no learning          0.90          0.875    0.864     9.48
              2. BEFORE - grouped CV, same month          0.96          0.975    0.921    10.11
3. AFTER  - trained Feb->Mar, deployed unchanged          0.96          0.980    0.925    10.11

base rate 0.095 | deploy rows 60,942 | clients 34
saved -> work/outputs/ml09_validation_audit_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### What is still open, said out loud

1. **One deployment window is not a backtest.** A real one repeats the train-then-deploy cycle
   across many decision moments. I have one.
2. **No calibration.** The scores order pages; nothing here shows that a 0.8 means an 80% chance.
3. **No causal evidence, and none is available** without an experiment in which some flagged pages
   are edited and others deliberately are not.
4. **June 2026 stays sealed** — it is the outcome window for any future forward test, and touching
   it now would spend evidence I will need later.

### What this audit changed about my own work

The paper's Finding #8 and my own population rule are the **same choice**: both study only the rows
that survived into the outcome window. Asking the question of the paper is what made me write it
into my own limitations rather than leave it in a code comment.